In [ ]:
dbutils.widgets.text("catalog_param", "my_assessment")
dbutils.widgets.text("schema_param", "gold")

catalog = dbutils.widgets.get("catalog_param")
schema = dbutils.widgets.get("schema_param")

In [ ]:
from pyspark.sql.functions import current_timestamp, col

# List of sample tables we are using
sample_tables = ["orders", "lineitem", "customer", "part", "supplier", "nation"]

for table in sample_tables:
    # 1. Read from the sample dataset
    df = spark.table(f"samples.tpch.{table}")
    
    # 2. Add Ingestion Metadata (Standard Bronze Practice)
    # This helps with auditing: "When did this data arrive?"
    df_bronze = df.withColumn("ingestion_date", current_timestamp()) \
                  .withColumn("source_path", col("_metadata.file_path"))
    
    # 3. Write to your Bronze Schema
    df_bronze.write.mode("overwrite").saveAsTable(f"my_assessment.bronze.{table}")
    
    print(f"Table {table} successfully ingested to Bronze.")